In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="evan0617/seniortalk", 
    repo_type="dataset", local_dir="./seniortalk", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 47 files: 100%|██████████| 47/47 [00:13<00:00,  3.59it/s]


'/home/ubuntu/seniortalk'

In [3]:
files = glob('seniortalk/*/*.parquet')
len(files)

47

In [4]:
df = pd.read_parquet(files[0])
df

,path,text
0,{'bytes': b'RIFF\xe4o\x00\x00WAVEfmt \x10\x00\...,嗯。
1,{'bytes': b'RIFF$r\x07\x00WAVEfmt \x10\x00\x00...,嗯，它知道哪个热乎，哪个凉，它不用你大人操心啊，小孩就要你大人操心啦。
2,{'bytes': b'RIFF$\x1d\x07\x00WAVEfmt \x10\x00\...,哎，你加衣服吧，啊天冷了，要加衣服，天热了，要脱衣服这个。
3,{'bytes': b'RIFFd\x13\x06\x00WAVEfmt \x10\x00\...,宠物就不同了，它不用你操心啊，就是你要跟它搞卫生，给它吃。
4,{'bytes': b'RIFFd\\\x00\x00WAVEfmt \x10\x00\x0...,嗯。
...,...,...
2949,{'bytes': b'RIFF\xe4b\x04\x00WAVEfmt \x10\x00\...,二百块钱那就真的现在一想起来那时候的二百块钱相当于现在你说。
2950,{'bytes': b'RIFF\xe45\x01\x00WAVEfmt \x10\x00\...,相不相于两千。
2951,{'bytes': b'RIFFd\xde\x00\x00WAVEfmt \x10\x00\...,都顶上万。
2952,{'bytes': b'RIFF\xa4j\x00\x00WAVEfmt \x10\x00\...,哎呀。


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['path'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = 20)

100%|██████████| 4/4 [01:57<00:00, 29.30s/it]


In [7]:
len(data)

53236

In [8]:
data[0]

{'audio_filename': 'seniortalk_audio/seniortalk-sentence_data-train-00011-of-00016_1.mp3',
 'text': '嗯，它知道哪个热乎，哪个凉，它不用你大人操心啊，小孩就要你大人操心啦。',
 'speaker': 'seniortalk_audio'}

In [9]:
with open('seniortalk.json', 'w') as fopen:
    json.dump(data, fopen)

In [10]:
audio_files = [d['audio_filename'] for d in data]

with open('seniortalk-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [14]:
# !zip -rq seniortalk_audio.zip seniortalk_audio

In [15]:
# !hf upload malaysia-ai/Multilingual-TTS seniortalk_audio.zip --repo-type=dataset

In [16]:
import json

with open('seniortalk.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 53236/53236 [00:00<00:00, 3149780.90it/s]


53236

In [18]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'seniortalk_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 53236/53236 [00:32<00:00, 1614.26it/s]


In [19]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [20]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'seniortalk_audio/seniortalk-sentence_data-train-00011-of-00016_1.mp3',
 'text': '嗯，它知道哪个热乎，哪个凉，它不用你大人操心啊，小孩就要你大人操心啦。',
 'speaker': 'seniortalk_audio_0'}

In [21]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'seniortalk')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 29.94ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  98%|█████████▊| 5.13MB / 5.25MB, 25.7MB/s  
Processing Files (1 / 1): 100%|██████████| 5.25MB / 5.25MB, 13.5MB/s  
Processing Files (1 / 1): 100%|██████████| 5.25MB / 5.25MB, 13.1MB/s  
New Data Upload: 100%|██████████| 5.25MB / 5.25MB, 13.1MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.16 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/07a7f9a8e0a1d4c79075f60629b93335b622f1d2', commit_message='Upload dataset', commit_description='', oid='07a7f9a8e0a1d4c79075f60629b93335b622f1d2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [7]:
# !zip -rq seniortalk_audio_neucodec.zip seniortalk_audio_neucodec

In [8]:
# !hf upload malaysia-ai/Multilingual-TTS seniortalk_audio_neucodec.zip --repo-type=dataset